In [1]:
from PySide2.QtCore import QTimer, QTime, Signal
import time
import numpy as np
import os
import sys

# Get the absolute path of the current script
script_path = os.path.abspath(r"C:\Users\YY3\GIT\squdi\src\qudi\jupyternotebooks\hom")
# Get the directory name of the script path
script_dir = os.path.dirname(script_path)
# Change the working directory to the script's directory
os.chdir(script_dir)
from hom import auto
from hom.auto import *
from hom.tools import *
%gui qt

In [2]:
folder_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\F-08\John\att1'
current_cryo = 'atto3' #fix the notation of current and non-active cryo
non_active_cryo = 'bf'
integrate_for_mins = 180
values = [0, -.2, -.5, -0.8, -1.13]  #the counts have to be equalized!

params = {
    'ple_gui': ple_gui,
    'laser_scanner_logic': laser_scanner_logic,
    'scanner_gui' : scanner_gui,
    'scanning_data_logic' : scanning_data_logic,
    'pulsestreamer' : pulsestreamer,
    'timetaggerlogic': timetaggerlogic,
    'timetagger': timetagger,
    'timetagger_remote': timetagger_remote,
    'timetaggerlogic_remote': timetaggerlogic_remote,
    'poi_manager_logic_remote': poi_manager_logic_remote,
    'poi_manager_logic': poi_manager_logic,
    'switchlogic': switchlogic,
    'ibeam_smart_remote': ibeam_smart_remote,
    'powercontroller_logic': powercontroller_logic,
    'integrate_for_mins': integrate_for_mins,
    'current_cryo': current_cryo,
    'non_active_cryo': non_active_cryo,
    'folder_save': folder_save,
    'values': values,
}

measurement_e_hom = auto.StarkHOM(ao_electrodes_remote, **params)  
measurement_e_hom.ao_electrodes._current_channel='ao3'
measurement_e_hom.ao_electrodes._constraints._channel_limits = {'ao3': [-1.38, 1.38]}
measurement_e_hom.min_position = 80
measurement_e_hom.max_position = 195
measurement_e_hom.max_power = 30e3
measurement_e_hom.perpendicular_position = 5
measurement_e_hom.parallel_position = 27
measurement_e_hom.polarization_is_parallel = False

measurement_e_hom.measurement_setpoints = [0, -0.2, -0.4, -0.6]
measurement_e_hom.to_refocus_ple = False

## Pre-align PLEs

In [23]:
measurement_e_hom.measurement_mode('Off-res')

In [22]:
measurement_e_hom.measurement_mode('PLE')

In [81]:
cts , setpoints = measurement_e_hom.align_resonances()

In [6]:
step_size = 0.02e-6

# Initialize target and current countsstep_size = step_size
target_position = measurement_e_hom.scanning_probe_logic_remote.scanner_target
target_counts = measurement_e_hom.get_counts(channels=[3]).sum()


current_counts = measurement_e_hom.get_counts(channels=[3]).sum()
    # Small step size for adjustments


In [7]:

direction_x = {'x': step_size, 'y': 0, 'z': 0}  # Initial direction for X adjustment
direction_y = {'x': 0, 'y': step_size, 'z': 0}  # Initial direction for Y adjustment
direction_z = {'x': 0, 'y': 0, 'z': step_size/2}  # Initial direction for Z adjustment
iteration = 0  # Tracks the current step within the 3-iteration cycle
initial_position = target_position.copy()
constraint_range = 0.6e-6  # 1 micrometer box around the initial position
constraints = {
    'x': (initial_position['x'] - constraint_range, 
          initial_position['x'] + constraint_range),
    'y': (initial_position['y'] - constraint_range, 
          initial_position['y'] + constraint_range),
    'z': (initial_position['z'] - constraint_range, 
          initial_position['z'] + constraint_range)
}

In [28]:
measurement_e_hom.scanning_probe_logic_remote.set_target_position(new_position)

{'x': 4.2209696743863595e-06, 'y': -5.5877e-06, 'z': 4.642e-06}

In [8]:
measurement_e_hom.get_counts(channels=[3]).sum()

10965.0

In [9]:
measurement_e_hom.get_counts(channels=[3]).sum()
for axis in target_position:
    # Perform 10-step sweep
    sweep_data = []
    for i in (x_data := np.arange(-6, 6, 0.5)):
        new_position = target_position.copy()
        new_position[axis] += float(i * step_size)
        new_position = {key : float(value) for key, value in new_position.items()}
        measurement_e_hom.scanning_probe_logic_remote.set_target_position(new_position)
        time.sleep(0.5)  # Adjust sleep time as needed
        sweep_data.append(measurement_e_hom.get_counts(channels=[3]).sum())

    # Fit a Gaussian model to the sweep data

    
    max_value, max_x = find_max_spline(x_data, sweep_data)

    # Update target position based on fitted maximum
    # target_position[axis] = max(self.constraints[axis][0], min(new_position[axis], self.constraints[axis][1]))
    target_position[axis] += max_x * step_size
    measurement_e_hom.scanning_probe_logic_remote.set_target_position(
        {key : float(value) for key, value in target_position.items()}
        )
    time.sleep(1)

In [4]:
measurement_e_hom.set_green_power('atto3', 
                                  25e3)

In [15]:
measurement_e_hom.ao_electrodes.setpoint = -0.7

### PLEs with green on

In [91]:
measurement_e_hom.set_green_power('bf',30e3)
measurement_e_hom.set_green_power('atto3', 
                                  30e3)

In [56]:
measurement_e_hom.ao_electrodes.setpoint = -1


In [20]:
measurement_e_hom.ao_electrodes.setpoint = -0.1

## Off-resonant refocus

In [39]:
pulsestreamer._seq.setDigital(3, [(100, 1)])
pulsestreamer._seq.setDigital(5, [(5000000, 1), (500000, 0)])
pulsestreamer.pulser_on()

0

In [19]:
measurement_e_hom.measurement_mode('PLE')

In [18]:
measurement_e_hom.measurement_mode('Off-res')

In [5]:
measurement_e_hom.refocus(optimize_both=True)

In [85]:
measurement_e_hom.parallel_position = 28

In [18]:
ibeam_smart_remote.power = 30e3

In [13]:
measurement_e_hom.polarization_is_parallel = True

In [ ]:
ibeam_smart_remote.power = 30e3

In [21]:
ibeam_smart_remote.power = 30e3

In [10]:
timetaggerlogic_remote.trace_data_avg[1][1][-5:].mean()

64629.0

In [88]:
measurement_e_hom.check_ple(do_ple_refocus = True)

In [19]:
measurement_e_hom.equalize_powers()

(10129.0, 8306.0, 16895.0)

### Fast step-wise PLE

15000.0

In [69]:
measurement_e_hom.set_green_power('atto3', 0)

In [ ]:
measurement_e_hom.to_refocus_ple = False
measurement_e_hom._ple_to_refocus = False

In [17]:
measurement_e_hom.to_refocus_ple = False
measurement_e_hom._ple_to_refocus = False
measurement_e_hom.check_ple(do_ple_refocus=False)

In [98]:
measurement_e_hom.to_refocus_ple = False
measurement_e_hom._ple_to_refocus = False
measurement_e_hom.hom_start_integration(integrate_for_mins = 600, 
                                        start_delay = 0)

In [6]:
measurement_e_hom.perpendicular_position = 5
measurement_e_hom.parallel_position = 28
measurement_e_hom.polarization_is_parallel = True

In [10]:
poi_manager_logic_remote._optimizelogic()._scan_logic().scanner_target

{'x': 4.581e-06, 'y': -5.7802e-06, 'z': 4.8984e-06}

In [4]:

iteration = 0  # Tracks the current step within the 3-iteration cycle
timer = QTimer()
measurement_e_hom.track_maximum(step_size = 0.05e-6)
timer.timeout.connect(measurement_e_hom.check_and_adjust_counts)
timer.start(2500)  # Check every 10 seconds

Current Counts: 4302.0, Target Counts: 4751.0
Moving to new position: {'x': 1.20192e-05, 'y': -5.5958e-06, 'z': 1.2218e-06}
New Counts: 4577.0, Target Position: {'x': 1.20192e-05, 'y': -5.5958e-06, 'z': 1.2218e-06}
Current Counts: 4751.0, Target Counts: 4751.0
Moving to new position: {'x': 1.20192e-05, 'y': -5.4958e-06, 'z': 1.2218e-06}
Target counts reached.
New Counts: 6854.0, Target Position: {'x': 1.20192e-05, 'y': -5.4958e-06, 'z': 1.2218e-06}
Current Counts: 7383.0, Target Counts: 6854.0
Moving to new position: {'x': 1.20192e-05, 'y': -5.4958e-06, 'z': 1.3218000000000002e-06}
Current Counts: 6446.0, Target Counts: 6854.0
Moving to new position: {'x': 1.20192e-05, 'y': -5.4958e-06, 'z': 1.1218e-06}
Target counts reached.
New Counts: 7151.0, Target Position: {'x': 1.20192e-05, 'y': -5.4958e-06, 'z': 1.1218e-06}
Current Counts: 7181.0, Target Counts: 7151.0
Moving to new position: {'x': 1.21192e-05, 'y': -5.4958e-06, 'z': 1.1218e-06}
New Counts: 6797.0, Target Position: {'x': 1.2019

In [4]:
measurement_e_hom.track_maximum()
# check_and_adjust_counts()

In [6]:
timer.stop()

In [35]:
measurement_e_hom.check_and_adjust_counts()

TypeError: StarkHOM.check_and_adjust_counts() missing 2 required positional arguments: 're' and 'm'

In [98]:
timer.stop()

In [82]:
measurement_e_hom.PLE_check_trigger(enable=False)

In [106]:
pulsestreamer._seq.setDigital(5, [(100, 0), (100, int(1))])
        
pulsestreamer.pulser_on()

0